# NiyamTrace-X — Native broker repair and adversarial verification

**CPU · standalone · no uploads · no paid keys**

Run all cells. This applies a verified source patch to the pinned repository, runs real SQLite security tests and saves actual traces and database snapshots. Controlled extraction fixtures are used; there is no live LLM inference. Failures are preserved in the output ZIP. The notebook creates a fresh working folder on every run.


In [ ]:
from pathlib import Path
import os,sys,subprocess,tempfile,json,hashlib,gzip,base64,zipfile,shutil
BASE_COMMIT = '651e87ef452c80008d4f73c8f2de21321e21fa50'
PATCHED_LOCAL_COMMIT = '8d1cb88de100c025800bc75cebd9bf1ed1fa5d59'
base_dir=Path('/content') if Path('/content').is_dir() else Path.cwd()
WORK=Path(tempfile.mkdtemp(prefix='NTX20_',dir=str(base_dir)))
REPO=WORK/'NiyamTrace-X'
OUT=WORK/'evidence';OUT.mkdir()
os.environ['LLM_BACKEND']='groq'
os.environ['GROQ_API_KEY']='offline-fixture-key-not-used'
os.environ['ENVIRONMENT']='test'
# Fixture-only provider is never called; no paid API credential is required.
def run(argv,cwd=None):
    result=subprocess.run(argv,cwd=cwd,text=True,capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stdout[-3000:]+'\n'+result.stderr[-3000:])
    return result
run(['git','clone','--depth','1','https://github.com/bnssaanirudh/NiyamTrace-X.git',str(REPO)])
if run(['git','rev-parse','HEAD'],REPO).stdout.strip()!=BASE_COMMIT:
    run(['git','fetch','--depth','1','origin',BASE_COMMIT],REPO)
run(['git','checkout','--detach',BASE_COMMIT],REPO)
assert run(['git','rev-parse','HEAD'],REPO).stdout.strip()==BASE_COMMIT
print('Pinned source:',BASE_COMMIT)
print('Fresh work directory:',WORK)


In [ ]:
PATCH_PAYLOAD = '...'
payload_bytes=gzip.decompress(base64.b64decode(PATCH_PAYLOAD))
assert hashlib.sha256(payload_bytes).hexdigest()=='388d54e4848d0013f8dac78f4b1b1bca021e0320394abc2402e0879a8fcde817'
patch_files=json.loads(payload_bytes)
for item in patch_files:
    rel=Path(item['path'])
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError('Unsafe source path')
    path=REPO/rel
    if item['original_sha256'] is None:
        if path.exists():raise RuntimeError('Unexpected pre-existing new file: '+str(rel))
    else:
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest()!=item['original_sha256']:
            raise RuntimeError('Original source hash mismatch: '+str(rel))
    data=base64.b64decode(item['base64'])
    assert hashlib.sha256(data).hexdigest()==item['patched_sha256']
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(data)
run(['git','diff','--check'],REPO)
(OUT/'SOURCE_PATCH_MANIFEST.json').write_text(json.dumps([{k:v for k,v in i.items() if k!='base64'} for i in patch_files],indent=2))
run([sys.executable,'-m','pip','install','--disable-pip-version-check','-q','-r',str(REPO/'niyamtrace/requirements-security.txt')])
run([sys.executable,'-m','pip','install','--disable-pip-version-check','-q','--no-deps','-e',str(REPO/'niyamtrace')])
run(['git','add','--all'],REPO)
run(['git','-c','user.name=NTX Colab Verification','-c','user.email=verification@local','commit','-m','Apply reviewed native broker security repair'],REPO)
SOURCE_COMMIT=run(['git','rev-parse','HEAD'],REPO).stdout.strip()
(OUT/'PATCHED_SOURCE_COMMIT.txt').write_text(SOURCE_COMMIT+'\n')
(OUT/'BASE_COMMIT.txt').write_text(BASE_COMMIT+'\n')
print('Patched files:',len(patch_files),'| local source commit:',SOURCE_COMMIT)


In [ ]:
# Launch tests as a subprocess so Colab's existing event loop is not reused.
command=[sys.executable,'-m','pytest','-q','--junitxml='+str(OUT/'full_suite_junit.xml')]
result=subprocess.run(command,cwd=REPO/'niyamtrace',text=True,capture_output=True)
(OUT/'full_suite_stdout.log').write_text(result.stdout)
(OUT/'full_suite_stderr.log').write_text(result.stderr)
TEST_RETURN_CODE=result.returncode
print(result.stdout[-4500:])
print('Full test return code:',TEST_RETURN_CODE)
# Errors are retained rather than turned into passes or hidden.


In [ ]:
env=os.environ.copy();env['PYTHONPATH']=str(REPO/'niyamtrace')
result=subprocess.run([sys.executable,str(REPO/'niyamtrace/scripts/run_broker_security_evidence.py'),'--out',str(OUT/'native_probes')],cwd=REPO/'niyamtrace',env=env,text=True,capture_output=True)
(OUT/'native_probe_stdout.log').write_text(result.stdout)
(OUT/'native_probe_stderr.log').write_text(result.stderr)
PROBE_RETURN_CODE=result.returncode
print(result.stdout[-4500:])
print('Native probe return code:',PROBE_RETURN_CODE)


In [ ]:
summary={'base_commit':BASE_COMMIT,'patched_source_commit':SOURCE_COMMIT,'full_suite_returncode':TEST_RETURN_CODE,'native_probe_returncode':PROBE_RETURN_CODE,'all_commands_passed':TEST_RETURN_CODE==0 and PROBE_RETURN_CODE==0,'study_type':'synthetic_native_source_security','live_model_inference':False,'human_authorization_evaluation':False,'database_backend':'SQLite'}
(OUT/'RUN_SUMMARY.json').write_text(json.dumps(summary,indent=2))
manifest={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.rglob('*') if p.is_file()}
(OUT/'ARTIFACT_SHA256.json').write_text(json.dumps(manifest,indent=2,sort_keys=True))
ZIP=WORK/'NTX20_NATIVE_BROKER_VERIFIED_RESULTS.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.rglob('*')):
        if path.is_file():z.write(path,path.relative_to(OUT))
print('Result archive:',ZIP)
print('All commands passed:',summary['all_commands_passed'])
print('Synthetic engineering evidence; human/model/official benchmark claims require separate experiments.')
try:
    from google.colab import files
    files.download(str(ZIP))
except ImportError:
    print('Outside Colab: open the displayed archive path.')
